In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from tqdm.auto import tqdm

In [38]:
SR = 16000
rng = np.random.default_rng(42)

# === configuración conservadora ===
N_VARIANTS_PER_CLIP = 1

P_STRETCH = 0.20
P_PITCH   = 0.00   # apagado por ahora
P_NOISE   = 0.50
P_REVERB  = 0.30

STRETCH_RANGE = (0.99, 1.01)
SNR_RANGE_DB  = (25.0, 35.0)
REVERB_RT60   = (0.10, 0.20)
REVERB_MIX    = (0.05, 0.10)

AUG_DIR = Path("/content/drive/MyDrive/pruebas/pre/train_aug/es")
AUG_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
def load_audio_mono_16k(path, target_sr=SR):
    y, sr = sf.read(str(path), always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    y = y.astype(np.float32, copy=False)
    if sr != target_sr:
        y = librosa.resample(y, orig_sr=sr, target_sr=target_sr, res_type="soxr_vhq")
        sr = target_sr
    return y, sr

def add_noise_snr(y, snr_db):
    sig_rms = np.sqrt(np.mean(y**2) + 1e-12)
    if sig_rms < 1e-6:
        return y
    noise = rng.standard_normal(len(y)).astype(np.float32)
    noise = noise / (np.sqrt(np.mean(noise**2)) + 1e-12)
    noise_rms = sig_rms / (10.0 ** (snr_db / 20.0))
    return (y + noise * noise_rms).astype(np.float32)

def add_reverb_simple(y, sr, rt60=0.20, mix=0.08):
    tau = rt60 / np.log(1000.0)
    n = int(rt60 * sr)
    if n < 1:
        return y
    t = np.arange(n) / sr
    h = np.exp(-t / tau).astype(np.float32)
    h = h / (np.sum(h) + 1e-12)
    y_rev = np.convolve(y, h, mode="full")[:len(y)]
    out = (1.0 - mix) * y + mix * y_rev
    return out.astype(np.float32)

def augment_one_safe(y, sr):
    ops = ["stretch", "noise", "reverb"]
    probs = np.array([P_STRETCH, P_NOISE, P_REVERB], dtype=float)
    probs = probs / probs.sum()

    op = rng.choice(ops, p=probs)

    if op == "stretch":
        rate = float(rng.uniform(*STRETCH_RANGE))
        y2 = librosa.effects.time_stretch(y, rate=rate)
        tag = f"ts{rate:.3f}"

    elif op == "noise":
        snr = float(rng.uniform(*SNR_RANGE_DB))
        y2 = add_noise_snr(y, snr)
        tag = f"ns{snr:.1f}"

    else:
        rt60 = float(rng.uniform(*REVERB_RT60))
        mix = float(rng.uniform(*REVERB_MIX))
        y2 = add_reverb_simple(y, sr, rt60=rt60, mix=mix)
        tag = f"rvb{rt60:.2f}s_{mix:.2f}"

    y2 = np.clip(y2, -0.999, 0.999).astype(np.float32)
    return y2, tag

In [39]:
df_tr = pd.read_csv("/content/drive/MyDrive/pruebas/audio_speaker/splits/train_fin.csv")
df_tr

,path,corpus,lang,speaker_id,clip_id,valence,arousal,dominance,id_audio,speaker_qu,speaker_id_final,group_id
0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_1-02-02-04,2.00,2.00,4.00,Audio10,NaN,Audio10,es_Audio10
1,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_16-02-02-04,2.00,2.00,4.00,Audio10,NaN,Audio10,es_Audio10
2,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_22-01-03-03,1.00,3.00,3.00,Audio10,NaN,Audio10,es_Audio10
3,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_23-02-02-04,2.00,2.00,4.00,Audio10,NaN,Audio10,es_Audio10
4,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_15-02-02-04,2.00,2.00,4.00,Audio10,NaN,Audio10,es_Audio10
...,...,...,...,...,...,...,...,...,...,...,...,...
10066,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,quechua,qu,10416,10416-3.5-2.75-2.5,3.50,2.75,2.50,10416,a6,a6,qu_a6
10067,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,quechua,qu,10417,10417-4.5-3.25-3.25,4.50,3.25,3.25,10417,a6,a6,qu_a6
10068,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,quechua,qu,10418,10418-4.25-3.5-3.75,4.25,3.50,3.75,10418,a1,a1,qu_a1
10069,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,quechua,qu,10419,10419-2.25-3.75-3.5,2.25,3.75,3.50,10419,a2,a2,qu_a2


In [40]:
# solo train español
df_train_es = df_tr[df_tr["lang"] == "es"].copy().reset_index(drop=True)

augmentation no es para balancear cantidad.

es para:

romper dependencia de speaker en español

Although multiple samples per speaker exist, speaker-level splitting ensures that the model is evaluated on unseen speakers, reducing the risk of identity-based overfitting.

In [41]:
rows_new = []

for _, row in tqdm(df_train_es.iterrows(), total=len(df_train_es), desc="Augment train-es"):
    in_path = Path(row["path"])
    y, sr = load_audio_mono_16k(in_path)

    stem = in_path.stem
    for k in range(N_VARIANTS_PER_CLIP):
        y_aug, ops = augment_one_safe(y, sr)
        out_name = f"{stem}_aug{k+1}_{ops}.wav"
        out_path = AUG_DIR / out_name
        sf.write(str(out_path), y_aug, sr, subtype="PCM_16")

        new_row = row.copy()
        new_row["path_norm"] = str(out_path)
        new_row["sr"] = sr
        new_row["is_aug"] = True
        new_row["aug_ops"] = ops
        rows_new.append(new_row)

df_aug_es = pd.DataFrame(rows_new)

Augment train-es:   0%|          | 0/1793 [00:00<?, ?it/s]

el modelo vera variabilidad intra-speaker natural

si el modelo memoriza voz → falla en test,
eso lo obliga a generalizar

In [42]:
# train final = original + augmentado
df_tr_aug = df_tr.copy()
df_tr_aug["path_norm"] = df_tr_aug["path"]
df_tr_aug["sr"] = SR
df_tr_aug["is_aug"] = False
df_tr_aug["aug_ops"] = ""

df_tr_aug = pd.concat([df_tr_aug, df_aug_es], ignore_index=True).reset_index(drop=True)

print("Train original:", len(df_tr))
print("Augmentados ES:", len(df_aug_es))
print("Train final:", len(df_tr_aug))
print(df_tr_aug["lang"].value_counts())

Train original: 10071
Augmentados ES: 1793
Train final: 11864
lang
qu    8278
es    3586
Name: count, dtype: int64


In [43]:
df_aug_es.head()

,path,corpus,lang,speaker_id,clip_id,valence,arousal,dominance,id_audio,speaker_qu,speaker_id_final,group_id,path_norm,sr,is_aug,aug_ops
0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_1-02-02-04,2.0,2.0,4.0,Audio10,NaN,Audio10,es_Audio10,/content/drive/MyDrive/pruebas/pre/train_aug/e...,16000,True,rvb0.14s_0.09
1,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_16-02-02-04,2.0,2.0,4.0,Audio10,NaN,Audio10,es_Audio10,/content/drive/MyDrive/pruebas/pre/train_aug/e...,16000,True,ns25.9
2,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_22-01-03-03,1.0,3.0,3.0,Audio10,NaN,Audio10,es_Audio10,/content/drive/MyDrive/pruebas/pre/train_aug/e...,16000,True,rvb0.13s_0.08
3,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_23-02-02-04,2.0,2.0,4.0,Audio10,NaN,Audio10,es_Audio10,/content/drive/MyDrive/pruebas/pre/train_aug/e...,16000,True,rvb0.12s_0.07
4,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_15-02-02-04,2.0,2.0,4.0,Audio10,NaN,Audio10,es_Audio10,/content/drive/MyDrive/pruebas/pre/train_aug/e...,16000,True,ns34.6


In [51]:
df_tr_aug[(df_tr_aug['id_audio']=='Audio10')&(df_tr_aug['is_aug']==True)]

,path,corpus,lang,speaker_id,clip_id,valence,arousal,dominance,id_audio,speaker_qu,speaker_id_final,group_id,path_norm,sr,is_aug,aug_ops
10071,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_1-02-02-04,2.0,2.0,4.0,Audio10,NaN,Audio10,es_Audio10,/content/drive/MyDrive/pruebas/pre/train_aug/e...,16000,True,rvb0.14s_0.09
10072,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_16-02-02-04,2.0,2.0,4.0,Audio10,NaN,Audio10,es_Audio10,/content/drive/MyDrive/pruebas/pre/train_aug/e...,16000,True,ns25.9
10073,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_22-01-03-03,1.0,3.0,3.0,Audio10,NaN,Audio10,es_Audio10,/content/drive/MyDrive/pruebas/pre/train_aug/e...,16000,True,rvb0.13s_0.08
10074,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_23-02-02-04,2.0,2.0,4.0,Audio10,NaN,Audio10,es_Audio10,/content/drive/MyDrive/pruebas/pre/train_aug/e...,16000,True,rvb0.12s_0.07
10075,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_15-02-02-04,2.0,2.0,4.0,Audio10,NaN,Audio10,es_Audio10,/content/drive/MyDrive/pruebas/pre/train_aug/e...,16000,True,ns34.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10169,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_90-02-02-03,2.0,2.0,3.0,Audio10,NaN,Audio10,es_Audio10,/content/drive/MyDrive/pruebas/pre/train_aug/e...,16000,True,ns31.7
10170,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_91-02-02-03,2.0,2.0,3.0,Audio10,NaN,Audio10,es_Audio10,/content/drive/MyDrive/pruebas/pre/train_aug/e...,16000,True,ns27.3
10171,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_96-02-02-04,2.0,2.0,4.0,Audio10,NaN,Audio10,es_Audio10,/content/drive/MyDrive/pruebas/pre/train_aug/e...,16000,True,ns25.8
10172,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_98-01-04-03,1.0,4.0,3.0,Audio10,NaN,Audio10,es_Audio10,/content/drive/MyDrive/pruebas/pre/train_aug/e...,16000,True,ns33.8


In [46]:
df_tr_aug['is_aug'].value_counts()

,count
is_aug,
False,10071
True,1793


In [47]:
df_tr_aug.to_csv("/content/drive/MyDrive/pruebas/audio_speaker/splits/aug/train_fin_aug.csv", index=False)

ver comportamiento:

In [33]:
STRETCH_RANGE = (0.99, 1.01)
SNR_RANGE_DB  = (25.0, 35.0)
REVERB_RT60   = (0.10, 0.20)
REVERB_MIX    = (0.05, 0.10)

In [25]:
df_test_aug = df_tr[df_tr["lang"] == "es"].sample(10, random_state=42)

In [32]:
import numpy as np

for i, row in df_test_aug.iterrows():
    path = row["path"]

    y, sr = load_audio_mono_16k(path)
    y_aug, ops = augment_one_safe(y, sr)

    L = min(len(y), len(y_aug))
    diff = np.mean(np.abs(y[:L] - y_aug[:L]))

    print("----")
    print("ops:", ops)
    print("diff:", diff)

----
ops: rvb0.15s_0.10
diff: 0.006639512
----
ops: ns28.4
diff: 0.0044803675
----
ops: ns23.7
diff: 0.0033643008
----
ops: ts0.981
diff: 0.08852153
----
ops: ts0.995
diff: 0.015929148
----
ops: ns25.2
diff: 0.004809267
----
ops: rvb0.19s_0.10
diff: 0.0063535655
----
ops: ts0.995
diff: 0.01975447
----
ops: rvb0.13s_0.07
diff: 0.003774008
----
ops: ns24.7
diff: 0.0016794814
